In [1]:
import sys
original_sys_path = sys.path.copy()
sys.path.append('../')
from utils_models import *

dq.set_device('cpu')
import warnings
warnings.filterwarnings("ignore")

In [2]:
solver = dq.solver.Tsit5(
                    rtol= 1e-06,
                    atol= 1e-06,
                    safety_factor= 0.9,
                    min_factor= 0.2,
                    max_factor = 5.0,
                    max_steps = int(1e4*1000),
                )

n_lvls_fluxonium = 20
n_lvls_transmon = 4


Ej_f = 2.7
Ec_f = 0.6
El_f = 0.13
qsf = qs.Fluxonium.create(
    n_lvls_fluxonium,
    {"Ej": Ej_f, "Ec": Ec_f, "El": El_f, "phi_ext": 0.0},
    N_pre_diag=100,
    use_linear = False
    )
g_tf = 0.2
Ec_t = 0.2
Ej_t =  3.40890048e+01

qst = MyTransmon.create(
    N = n_lvls_transmon,
    params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
    N_max_charge=10
    )


devices = [qsf, qst]
f_indx = 0
t_indx = 1
Ns = [device.N for device in devices]
fn = qs.promote(qsf.ops["n"], f_indx, Ns)
tn = qs.promote(qst.ops['n'], t_indx, Ns)

system = qs.System.create(devices, couplings=[
    g_tf *  fn @ tn
    ])
system.params["g_tf"] = g_tf
system_evals_sorted, system_evecs_sorted, product_indices_sorted_by_eval = calculate_eig(Ns, system.get_H())
system_evals_in_product_indices, system_evecs_in_product_indices = system.calculate_eig_linear()

driven_op = transform_op_into_dressed_basis_jax(tn, system_evecs_sorted.T)

def truncate(data: jnp.array):
    return data[:,:]

tot_dim = n_lvls_fluxonium * n_lvls_transmon

diag_hamiltonian = 2 * jnp.pi *truncate(jnp.diag(system_evals_sorted))
psi0_list = [truncate(dq.basis(tot_dim,find_closest_dressed_index(l*qst.N, product_indices_sorted_by_eval)))
                    for l in [0,1,2]] #00,10,20

zero_one_op = dq.basis_dm(tot_dim, find_closest_dressed_index(0*qst.N+1, product_indices_sorted_by_eval))
one_one_op = dq.basis_dm(tot_dim, find_closest_dressed_index(1*qst.N+1, product_indices_sorted_by_eval))
two_one_op = dq.basis_dm(tot_dim, find_closest_dressed_index(2*qst.N+1, product_indices_sorted_by_eval))
pulse_length = 400
tlist = jnp.linspace(0,pulse_length,int(pulse_length))


In [3]:
system_evals_in_product_indices[0,1] - system_evals_in_product_indices[0,0], system_evals_in_product_indices[1,1] - system_evals_in_product_indices[1,0], system_evals_in_product_indices[2,1] - system_evals_in_product_indices[2,0]

(Array(7.175811, dtype=float64),
 Array(7.1835878, dtype=float64),
 Array(7.18356691, dtype=float64))

In [4]:

def objective(params):    
    sigma = params[0]
    amp_with_2pi = params[1]
    w_d = params[2]
    beta = params[3]
    
    pulse_shape_args={
        'w_d': w_d,
        'amp': amp_with_2pi/(2*jnp.pi),
        'duration': pulse_length,
        'sigma': sigma,
        'beta':beta
    }      

    def _H(t):
        _H =  diag_hamiltonian +  truncate(driven_op) * modified_drag_pulse(t, pulse_shape_args)
        return _H 
    H =  dq.timecallable(_H)
    result = dq.sesolve(
                H = H,
                psi0 = psi0_list,
                tsave = tlist,
                solver = solver
                )
        
    f0_e = dq.expect(zero_one_op,
                    result.states[0][-1]).real
    f1_e = dq.expect(one_one_op,
                    result.states[1][-1]).real
    f2_e = dq.expect(two_one_op,
                    result.states[2][-1]).real
    return 1 - f0_e + f1_e + f2_e
    # return f0_e

func = jax.value_and_grad(objective)

In [5]:
sigma = 2.47029937e+02 / np.sqrt(2*np.pi)
amp_with_2pi = 9.13069185e-03
w_d = 7.175811
beta = 0.0

params =  jnp.array([
                sigma,
                amp_with_2pi,
                w_d,
                beta
            ])



optimizer = optax.nadam(learning_rate=jnp.array([
                                                1e-2,
                                                 4e-3,
                                                 1e-4,
                                                 1e-3
                                                 ])) 
opt_state = optimizer.init(params)

num_steps = 1000
for step in range(num_steps):
    val, grads = func(params)
    print(f"iter: {step}, val={val} grads: {grads}, params: {params}")
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f'Optimized params: {params}')

iter: 0, val=0.055801201307798656 grads: [-2.66065266e-03 -6.18840782e+01 -4.64110862e+01 -6.33443311e-04], params: [9.85506864e+01 9.13069185e-03 7.17581100e+00 0.00000000e+00]
iter: 1, val=0.37370199633350026 grads: [ 1.04135533e-02  1.44663802e+02 -8.43739924e+00  3.73863765e-05], params: [9.85654232e+01 1.50254287e-02 7.17595837e+00 1.47366095e-03]
iter: 2, val=0.013142894461479393 grads: [1.53668305e-03 2.78478344e+01 5.02648946e+01 9.64831227e-04], params: [9.85547099e+01 1.12264753e-02 7.17602169e+00 1.82417826e-03]
iter: 3, val=0.031202446706418197 grads: [-2.00849325e-03 -4.48333877e+01  3.28132911e+01  6.11943532e-04], params: [9.85501506e+01 9.55626623e-03 7.17597174e+00 1.09980915e-03]
iter: 4, val=0.026820794984131622 grads: [-1.89229208e-03 -4.17586308e+01  8.35039388e+00  2.37889889e-04], params: [9.85488715e+01 9.63070516e-03 7.17592522e+00 4.74849694e-04]
iter: 5, val=0.010310129730782724 grads: [-9.98198200e-04 -2.12024865e+01 -1.03956050e+01 -2.36012712e-05], params:

KeyboardInterrupt: 